### Desigualdad en Chile: un estudio enfocado en las comunas y su acceso a hospitalizaciones complejas
Integrantes: Ignacia Castillo, Amalia Díaz, Juan Pablo Molina, María Poddubnaya <br>
<br>
Se busca estudiar la desigualdad en el acceso a hospitalización compleja. Para esto, se analizan egresos hospitalarios de la base de datos del GRD, enfocándonos en casos de alta complejidad, definidos como egresos con severidad 2 y 3 (Moderada y Mayor) y peso GRD > 2 (casos que cuesten el doble al promedio y más).
Además, para entender el impacto de la desigualdad en estos registros, se trabaja con los siguientes datasets:
- CIE-10 (tabla maestra de códigos de diagnósticos)
- Pobreza comunal 2024 (MINDESOF, índice pobreza multidimensional comunal)
- Base de Establecimientos 2024 (DIES, asocia código hospital con datos específicos)
- Dotación de camas hospitalarias (DIES, camas totales y camas criticas por año y establecimiento)
- Proyección de poblacion 2002-2035 (INE, versión 2017, obtener población 2018-2024 por comuna)

Antes de comenzar, se importan librerías necesarias:

In [1]:
import pandas as pd
import numpy as np
import time
import unicodedata

#### Cruce de datos: preparación
Luego, comienza la carga de bases de datos para cruzar. Primero, la del CIE-10, donde se utilizará el codigo (ej.: A12.30), presente en la base del GRD, para cruzar los datos y añadir la descripción y sección del diagnóstico (hay casi 40.000 diagnósticos diferentes).

In [2]:
# se declaran las columnas a importar
cols_cie10 = ['Código', 'Descripción', 'Sección']
# lectura de datos
ruta = 'datasets/grd/CIE-10.xlsx'
dtypes = {col:'string' for col in cols_cie10}
cie10 = pd.read_excel(ruta, usecols=cols_cie10, dtype=dtypes)

# se renombran columnas para calzar con el estilo del GRD
cie10.rename(columns={
    'Código':'DIAGNOSTICO1',
    'Descripción':'DESCRIPCION',
    'Sección':'SECCION'
    }, inplace=True)
# información básica sobre valores importados
cie10.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39873 entries, 0 to 39872
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   DIAGNOSTICO1  39873 non-null  string
 1   DESCRIPCION   39873 non-null  string
 2   SECCION       39873 non-null  string
dtypes: string(3)
memory usage: 934.7 KB


Para la base de pobreza comunal 2024, se toma el índice de pobreza multidimensional por comuna, que es simplemente el porcentaje de población de la comuna viviendo en pobreza multidimensional (que considera Educación, Salud, Trabajo/Seguridad Social, Vivienda/Entorno y Redes/Cohesión Social)

In [3]:
# se cargan datos del excel
ruta = 'datasets/sae_multidimensional_2024.xlsx'
dtypes_pm = {
    'Porcentaje de personas en situacion de pobreza multidimensional 2024':'float32',
    'Código':'string'
}
cols_pm = list(dtypes_pm.keys())
pm = pd.read_excel(ruta, header=2, usecols=cols_pm, dtype=dtypes_pm)
# se ajusta el codigo de comuna, para hacer match con el GRD
pm['Código'] = pm['Código'].str.strip().str.zfill(5)
# renombre de comunas para calzar con GRD
pm.rename(columns={
    'Porcentaje de personas en situacion de pobreza multidimensional 2024':'POB_MULTIDIM',
    'Código':'COD_COMUNA'
    }, inplace=True)
# pm['COMUNA_REGISTRADA'] = pm['COMUNA_REGISTRADA'].str.strip().str.upper().apply(remove_tildes)
pm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 345 entries, 0 to 344
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   COD_COMUNA    345 non-null    string 
 1   POB_MULTIDIM  345 non-null    float32
dtypes: float32(1), string(1)
memory usage: 4.2 KB


Con tal de poder trabajar a fondo con las comunas, dado el enfoque de nuestro trabajo, es importante poder estandarizar los nombres en cualquier dataset que contenga las comunas. En este caso, se utilizará la función de remover tildes y luego todas las comunas en mayúscula para hacer que calcen.

In [4]:
def remove_tildes(text):
    return unicodedata.normalize("NFD", text).encode("ascii", "ignore").decode("utf-8")

Para el dataset de la base de establecimientos, se usa el código de la comuna como columna compartida, y se traen códigos de región y sus nombres, junto con datos clave de cada establecimiento de salud: ubicación, nivel de complejidad del establecimiento, etc.

In [5]:
# se declara la ruta, columnas y sus dtypes (tipo de dato) para cargar excel
ruta = "datasets/base establecimientos/Base de Establecimientos 2024.xlsx"
dtypes_estab = {'Código Vigente':'string',
        'Código Región':'str',
        'Nombre Región':'category',
        'Código Comuna':'str',
        'Nombre Comuna':'category',
        'Nombre Oficial':'string',
        'Tipo de Prestador Sistema de Salud': 'string',
        'Nivel de Complejidad': 'category',
        'LATITUD      [Grados decimales]':'float32',
        'LONGITUD [Grados decimales]':'float32',
        }
cols_estab = list(dtypes_estab.keys())
# se lee el excel y especifica que Pendiente es un valor nulo para la ubicacion
estab = pd.read_excel(ruta, header=1, usecols=cols_estab, dtype=dtypes_estab, 
                                na_values={"LATITUD      [Grados decimales]": "Pendiente",
                                "LONGITUD [Grados decimales]": "Pendiente"}, decimal=',')

# se asegura que los codigos sean string para modificarlos y que calcen con GRD
estab['Código Región'] = estab['Código Región'].astype('string')
estab['Código Comuna'] = estab['Código Comuna'].astype('string')
estab['Código Vigente'] = estab['Código Vigente'].astype('string')
# se renombran columnas para mantener estilo GRD
estab.rename(columns={
        "Código Vigente":"COD_ESTABLECIMIENTO",
        'Código Región':'COD_REGION',
        'Código Comuna':'COD_COMUNA',
        'Nombre Región':'REGION',
        'Nombre Comuna':'COMUNA',
        'Nombre Oficial':'ESTABLECIMIENTO',
        'Tipo de Prestador Sistema de Salud':'PRESTADOR',
        'Nivel de Complejidad':'COMPLEJIDAD_ESTAB',
        'LATITUD      [Grados decimales]':'LATITUD',
        'LONGITUD [Grados decimales]':'LONGITUD',
        }, inplace=True)
# se normalizan los codigos para tener 5 digitos, y si tienen 4, partir con un 0
estab['COMUNA'] = estab['COMUNA'].str.strip().str.upper().apply(remove_tildes)
estab['COD_COMUNA'] = estab['COD_COMUNA'].str.strip().str.zfill(5)
estab['PRESTADOR'] = estab['PRESTADOR'].str.strip()
estab['ESTABLECIMIENTO'] = estab['ESTABLECIMIENTO'].str.strip()
estab = estab.dropna().reset_index(drop=True)
display(estab.shape, estab.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4585 entries, 0 to 4584
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   COD_ESTABLECIMIENTO  4585 non-null   string  
 1   COD_REGION           4585 non-null   string  
 2   REGION               4585 non-null   category
 3   ESTABLECIMIENTO      4585 non-null   string  
 4   COD_COMUNA           4585 non-null   string  
 5   COMUNA               4585 non-null   object  
 6   LATITUD              4585 non-null   float32 
 7   LONGITUD             4585 non-null   float32 
 8   PRESTADOR            4585 non-null   string  
 9   COMPLEJIDAD_ESTAB    4585 non-null   category
dtypes: category(2), float32(2), object(1), string(5)
memory usage: 260.7+ KB


(4585, 10)

None

Luego, se importa un dataset complementario, que son las camas hospitalarias a los establecimientos. Este dataset y el anterior son AMBOS del DEIS, por lo que sus datos son similares y se importa con el mismo código como variable que une estos datasets.

In [6]:
# nuevamente, se declara la ruta
ruta = 'datasets/Dotacion_Camas_Hospitalarias_2025.xlsx'
# esta vez, las columnas con la cantidad de camas tienen el año como nombre,
# se importan desde 2019-2024, para cruzar segun año
dtypes_camas = {
    'Código Establecimiento':'string',
    'Área Funcional':'string',
    **{year:'string' for year in range(2019,2025)}
}
cols_camas = list(dtypes_camas.keys())
# al importar, las columnas deben leerse con una funcion lambda, ya que al ser numeros no se leen correctamente
camas = pd.read_excel(ruta, header=2, dtype=dtypes_camas, usecols=lambda col: col in cols_camas)
# se renombran columnas para calzar con GRD
camas.rename(columns={
    year:f'{year}' for year in range(2019,2025)}, inplace=True)
camas.rename(columns={
    'Código Establecimiento':'COD_ESTABLECIMIENTO',
    'Área Funcional':'Area'
}, inplace=True)

# el formato del documento incluye codigos en una columna que representan a muchas filas,
# por lo que se realiza un forward fill para que se repita y sea posible agrupar luego
camas['COD_ESTABLECIMIENTO'] = camas['COD_ESTABLECIMIENTO'].ffill()
# camas = camas.dropna(
#     subset= ['COD_ESTABLECIMIENTO']
#     ).reset_index(drop=True)

# hay muchos tipos de cama, por lo que se guardan los totales por separado
totales = camas[camas['Area'] == 'Total'].copy()

# las camas criticas se agrupan, para considerarlas en una sola variable
camas_criticas = [
    'Área Cuidados Intensivos Adultos',
    'Área Cuidados Intermedios Adulto',
    'Área Cuidados Intensivos Pediátrica',
    'Área Cuidados Intermedios Pediátricos',
    'Área Neonatología Cuidados Intensivos',
    'Área Neonatología Cuidados Intermedios'
]
# se filtran datos para incluir estos valores en una variable aparte
criticas = camas[camas['Area'].isin(camas_criticas)].copy()
# se elimina columna de area, ahora innecesaria
totales = totales.drop(columns=['Area'])
criticas = criticas.drop(columns=['Area'])

# se convierten los datos a numeros, dejando valores vacios como 0
for year in range(2019,2025):
    criticas[f'{year}'] = pd.to_numeric(criticas[f'{year}'], errors='coerce')
    totales[f'{year}'] = pd.to_numeric(totales[f'{year}'], errors='coerce')
    criticas[f'{year}'] = criticas[f'{year}'].fillna(0).astype('UInt16')
    totales[f'{year}'] = totales[f'{year}'].fillna(0).astype('UInt16')
cols_camas_str = [f'{year}' for year in range(2019,2025)]

# se agrupan las camas criticas para que quede la suma de ellas por cada establecimiento
criticas = pd.DataFrame(criticas.groupby('COD_ESTABLECIMIENTO', as_index=False).sum())
# como hay datos de muchos años, para poder cruzar los datos considerando los años, es necesario
# que sean parte de las filas, con una columna unica de año, por lo que se hace melt de los datos
criticas = criticas.melt(
    id_vars='COD_ESTABLECIMIENTO',
    value_vars=cols_camas_str,
    var_name='YEAR',
    value_name='CAMAS_CRITICAS'
)
# lo mismo para las camas totales, dejando todos los años en una sola columna
totales = totales.melt(
    id_vars='COD_ESTABLECIMIENTO',
    value_vars=cols_camas_str,
    var_name='YEAR',
    value_name='CAMAS_TOTALES'
)
# finalmente, se cruzan los datos de camas criticas y totales, para cargar datos de camas por establecimiento
camas = pd.merge(totales, criticas, on=['COD_ESTABLECIMIENTO','YEAR'], how='left')
camas['CAMAS_CRITICAS'] = camas['CAMAS_CRITICAS'].fillna(0)
camas['YEAR'] = camas['YEAR'].astype('int32')
display(camas.info(), camas.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1182 entries, 0 to 1181
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   COD_ESTABLECIMIENTO  1182 non-null   string
 1   YEAR                 1182 non-null   int32 
 2   CAMAS_TOTALES        1182 non-null   UInt16
 3   CAMAS_CRITICAS       1182 non-null   UInt16
dtypes: UInt16(2), int32(1), string(1)
memory usage: 20.9 KB


None

,COD_ESTABLECIMIENTO,YEAR,CAMAS_TOTALES,CAMAS_CRITICAS
0,101100,2019,313,38
1,102100,2019,440,60
2,201319,2019,0,0
3,103103,2019,40,0
4,103104,2019,10,0


Para el último dataset a cruzar, se tiene la proyección de población por año, a nivel de comuna, provincia o región dado por el INE en 2017. Se utilizará para contar con valores de población específicos según el año de egreso hospitalario GRD.

In [7]:
# se cargan region, comuna y años, donde está el valor de la población
ruta = 'datasets/Proyeccion_2002-2035_Comunas.xlsx'
# similarmente a las camas, se consideran años 2019-2024
dtypes_pob = {
    'Region':'string',
    'Comuna':'string',
    **{f'Poblacion {year}':'uint16' for year in range(2019,2025)}
}
cols_pob = list(dtypes_pob.keys())
# se lee el excel con columnas establecidas
poblacion = pd.read_excel(ruta, usecols=cols_pob, dtype=dtypes_pob)
# igual que con las camas, se necesita esta información según año en UNA sola columna,
# por lo que se hace un melt de los datos, enfocandose en los años.
poblacion = poblacion.melt(
    id_vars=['Region','Comuna'],
    value_vars=[f'Poblacion {year}' for year in range(2019,2025)],
    var_name='YEAR',
    value_name='Poblacion'
)
# dentro del valor del año, se remueve la palabra población, dejando solo el valor numérico
poblacion['YEAR'] = poblacion['YEAR'].str.replace('Poblacion ', '').astype('int32')
# se agrupan valores según region, comuna, y año primero, en variable aparte
comunas = poblacion.groupby(
    by=['Region','Comuna','YEAR'],
    observed=True
    )['Poblacion'].sum().reset_index() # y se calcula la suma, en este caso es comunal
comunas.rename(columns={'Poblacion':'POB_COMUNA'}, inplace=True)

# se hace el mismo proceso, pero ahora solo con la región, para contar con ambos datos
regiones = comunas.groupby(
    by=['Region','YEAR'],
    observed=True
    )['POB_COMUNA'].sum().reset_index() # se obtienen totales regionales de población, por año
regiones.rename(columns={'POB_COMUNA':'POB_REGION'}, inplace=True)

# finalmente, se unen estas dos variables separadas, para tener un solo DataFrame con toda la información
poblacion = pd.merge(
    comunas, 
    regiones, 
    on=['Region','YEAR'],
    how='left'
)
# igual que antes, se rellena el codigo con 0s en caso de no tener 5 digitos, para cruzar con GRD
poblacion['Comuna'] = poblacion['Comuna'].str.strip().str.zfill(5)
poblacion.rename(columns={'Comuna':'COD_COMUNA', 'Region':'COD_REGION'}, inplace=True)
display(poblacion.head(), poblacion.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2076 entries, 0 to 2075
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   COD_REGION  2076 non-null   string
 1   COD_COMUNA  2076 non-null   string
 2   YEAR        2076 non-null   int32 
 3   POB_COMUNA  2076 non-null   uint64
 4   POB_REGION  2076 non-null   uint64
dtypes: int32(1), string(2), uint64(2)
memory usage: 73.1 KB


,COD_REGION,COD_COMUNA,YEAR,POB_COMUNA,POB_REGION
0,1,01101,2019,216514,368906
1,1,01101,2020,223463,382773
2,1,01101,2021,227127,391165
3,1,01101,2022,229072,396697
4,1,01101,2023,230595,401588


None

Ahora, se realiza un proceso un poco más largo para cargar la base de datos GRD, que contempla desde los años 2019 al 2024, y cambios año a año, por lo que se utilizan ciertas funciones que faciliten el proceso y logren consolidar toda la información en un único dataset.

In [8]:
# codificaciones del texto a intentar, dado que hubo problemas al usar utf-8 o latin inicialmente
ENCODINGS_TO_TRY = ['utf-8-sig', 'utf-16', 'utf-8', 'cp1252', 'latin1']
# se probara primero leer algunos caracteres y revisar la codificación, y en caso de haber error
# intentar la siguiente
def detect_encoding(filepath):
    for enc in ENCODINGS_TO_TRY:
        try:
            with open(filepath, 'r', encoding=enc) as f:
                f.read(8192)
            return enc
        except (UnicodeDecodeError, UnicodeError):
            continue
    return 'latin1'  # en caso de que todo falle, se ocupa una confiable
# se incluyen todas las variables a cargar para el GRD, y sus dtypes
dtypes_grd = {'COD_HOSPITAL':"string", 
        'CIP_ENCRIPTADO':'string',
        'SEXO':"category",
        'PREVISION':"category",
        'COMUNA':"category",
        'PROVINCIA':"category",
        'TIPOALTA':"category",
        'SERVICIO_SALUD':"category",
        'TIPO_PROCEDENCIA':"category",
        'TIPO_INGRESO':"category",
        **{f'DIAGNOSTICO{i}':"string" for i in range(1,36)},
        **{f'PROCEDIMIENTO{i}':"string" for i in range(1,31)},
        'IR_29301_COD_GRD':'string',
        'IR_29301_PESO':'string',
        'IR_29301_SEVERIDAD':"category",
        'IR_29301_MORTALIDAD':"category"
        }

target_cols = list(dtypes_grd.keys())
# estas columnas especificas contienen fechas, se usaran luego
cols_fechas = ['FECHA_NACIMIENTO', 'FECHA_INGRESO', 'FECHAALTA']
target_cols.extend(cols_fechas)
# se utiliza un diccionario con parametros clave para la carga del grd en pd.read_csv()
config_carga = {
        # con prueba y error, se descubre que el separador es "|" para cada archivo .txt de GRD
        "sep": "|",
        # se leeran fechas con su dtype apropiado segun columnas antes establecidas
        "parse_dates": cols_fechas,
        # con muchos datos, se especifica una opción de memoria
        "low_memory":False,
        # columnas objetivo
        "usecols": target_cols,
        # en caso de haber lineas con errores, saltarlas
        "on_bad_lines": "skip",
        # tipo de datos
        "dtype": dtypes_grd,
}

Luego, se importan los archivos de cada año, registrando tiempos de carga.

In [9]:
# cada uno de los grd, segun el año, se guardará en este diccionario
grds = {}
# comienza la carga de todos los datos
for year in range(2019,2025):
    print(f'abriendo grd{year}...')
    # se leen los parametros
    params = config_carga.copy()
    # y las columnas a cargar
    cols = target_cols.copy()
    # se especifica la ruta segun el año
    ruta = f"datasets/grd/GRD_PUBLICO_{year}.txt"
    # y la codificación más probable según la funcion
    encoding = detect_encoding(ruta)
    # se registra el tiempo de carga, solo por motivos informativos
    inicio = time.time()
    # para 2024, la columna del rut encriptado cambia de nombre, por lo que se considera
    if year == 2024:
        cols[1] = 'ID_BENEFICIARIO'
        params['usecols'] = cols
    # se lee el archivo
    grds[f'grd{year}'] = pd.read_csv(ruta, encoding = encoding, **params)
    # y se registra tiempo de finalizacion
    final = time.time()
    # para informar de carga 
    print(f'carga de grd{year} en {final-inicio:.3f}s\n')

abriendo grd2019...
carga de grd2019 en 17.523s

abriendo grd2020...
carga de grd2020 en 9.718s

abriendo grd2021...
carga de grd2021 en 11.060s

abriendo grd2022...
carga de grd2022 en 14.219s

abriendo grd2023...


/tmp/ipykernel_9157/1221887794.py:21: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  grds[f'grd{year}'] = pd.read_csv(ruta, encoding = encoding, **params)


carga de grd2023 en 13.933s

abriendo grd2024...
carga de grd2024 en 14.778s



Una vez cargados, es necesario estandarizar las fechas y juntarlos en un solo dataset.

In [57]:
# se estandariza la fecha de 2023, que tiene ciertas columnas con dias primero
grds['grd2023']['FECHAALTA'] = pd.to_datetime(grds['grd2023']['FECHAALTA'], dayfirst=True, errors='coerce')
grds['grd2023']['FECHA_INGRESO'] = pd.to_datetime(grds['grd2023']['FECHA_INGRESO'], dayfirst=True, errors='coerce')
# se renombra la columna modificada en 2024 al nombre antiguo
grds['grd2024'].rename(columns={"ID_BENEFICIARIO":"CIP_ENCRIPTADO"}, inplace=True)
# se unen los datasets de todos los años, tras estandarizar los formatos
grd = pd.concat(grds.values(), ignore_index=True)
# se pasan las columnas a dtype datetime64
for col in cols_fechas:
    grd[col] = pd.to_datetime(grd[col], errors='coerce')

Después, se apunta a ciertas columnas con problemas. En este caso, el peso GRD representa una proporción de costo del egreso según promedio. Por ejemplo, una estadía que cuesta el triple que el promedio, tiene un peso GRD de 3, una que cuesta el promedio es peso 1, y una que cuesta la mitad tiene un valor de 0.5. Al revisar un poco los datos, se encuentran muchos valores como 12812, 23319, 14012, en los que los costos no tendrían sentido, con no solo una persona costando 20 mil veces mas que el promedio, sino múltiples. Se asume que es un error de tipeo/carga/lectura de datos, y que deberían tener una coma. Existe la posibilidad de GRDs que sean mayor a 10, por lo que 14012 -> 1.4012 o 14.012 son ambos válidos. En este caso, se asumió, para simplificar el problema, que tienen la coma al principio, tal que: 14012 -> 1.4012, aunque se entiende que esto introduce un posible sesgo.

In [58]:
# se lee el rut encriptado como dtype string, para que no se considere un
# numero y aparezca en las estadisticas erroneamente
grd['CIP_ENCRIPTADO'] = grd['CIP_ENCRIPTADO'].astype('string')

# se cambia la coma por punto para interpretar como float
grd['IR_29301_PESO'] = grd.IR_29301_PESO.str.replace(',','.', regex=False)
# los que NO tienen punto
dec_coma = ~grd['IR_29301_PESO'].str.contains('.', regex=False)
# todos como float
floats = pd.to_numeric(grd['IR_29301_PESO'], errors='coerce')
# grd['IR_29301_PESO'] = grd.IR_29301_PESO.str.replace('.','')
powers = np.maximum(grd['IR_29301_PESO'].str.len() - 1,1)
# se divide solo donde no hay punto, por error de tipeo
grd['IR_29301_PESO'] = np.where(dec_coma, floats / (10 ** powers), floats)

# columnas categoricas, que guardan valores unicos, o categorias de pandas
categ_cols = ['SEXO', 'PREVISION', 'TIPO_PROCEDENCIA', 'TIPO_INGRESO', 'IR_29301_SEVERIDAD', 'IR_29301_MORTALIDAD',
            'TIPOALTA', 'PROVINCIA', 'COMUNA','SERVICIO_SALUD']
# columnas de diagnosticos y procedimientos
diag_cols = [f'DIAGNOSTICO{i}' for i in range(2,36)]
proced_cols = [f'PROCEDIMIENTO{i}' for i in range(1,31)]
# se eliminan nulos tras estandarizar la mayoria de columnas
cols_ignore = diag_cols + proced_cols
grd = grd.dropna(subset=[col for col in grd.columns if col not in cols_ignore])
# se transforman las columnas anteriores a categoricas
grd = grd.astype({col: 'category' for col in categ_cols})
# se reescriben ciertos nombres de columna para calzar con los merges proximos
grd.rename(columns={'COMUNA':'COMUNA_REGISTRADA', 'COD_HOSPITAL':'COD_ESTABLECIMIENTO'}, inplace=True)
# se filtran datos para que solo sean con fecha ingreso >= 2019
grd['YEAR'] = grd['FECHA_INGRESO'].dt.year
grd['YEAR'] = grd['YEAR'].astype('int32')
grd = grd[grd['YEAR'] >= 2019]
grd.reset_index(drop=True, inplace=True)

# se estandarizan las comunas para que la registrada y del establecimiento
# esten escritas de la misma manera y puedan compararse
grd['COMUNA_REGISTRADA'] = grd['COMUNA_REGISTRADA'].str.strip().str.upper().apply(remove_tildes)
grd['COMUNA_REGISTRADA'] = grd['COMUNA_REGISTRADA'].str.replace('COIHAIQUE','COYHAIQUE')
grd['COMUNA_REGISTRADA'] = grd['COMUNA_REGISTRADA'].str.replace('AISEN','AYSEN')
grd['COMUNA_REGISTRADA'] = grd['COMUNA_REGISTRADA'].str.replace('CON CON','CONCON')
grd['COMUNA_REGISTRADA'] = grd['COMUNA_REGISTRADA'].str.replace('O"HIGGINS','OHIGGINS')

# se crean variables clave, edad y dias estancia
grd['EDAD'] = ((grd['FECHA_INGRESO'] - grd['FECHA_NACIMIENTO']).dt.days / 365.25).round(1)
grd['DIAS_ESTANCIA'] = (grd['FECHAALTA'] - grd['FECHA_INGRESO']).dt.days
# se arreglan ciertas columnas y sus tipos de dato
grd['IR_29301_SEVERIDAD'] = grd.IR_29301_SEVERIDAD.astype('string').astype('category')

# para los diagnosticos, los 'DESCONOCIDO' se reemplazan con nulo, asumiendo que esto reduce
# el numero de diagnosticos en una cantidad reducida de pacientes
for col in diag_cols:
    grd[col] = grd[col].astype('string')
    grd[col].replace({'DESCONOCIDO':np.nan})
    grd[col] = grd[col].astype('category')
# se cuenta la cantidad total de comorbilidades y no. de procedimientos, para añadirlos
# como variable al dataset
grd['N_COMORBILIDADES'] = grd[diag_cols].count(axis=1)
grd['N_PROCEDIMIENTOS'] = grd[proced_cols].count(axis=1)

El índice de comorbilidad de Charlson va más allá de contar la cantidad de comorbilidades, sino que le da un peso a cada una según su mortalidad a largo plazo. Se incluye el diccionario que mapea pesos a diagnósticos aquí abajo, para poder introducir esta variable al dataset (crédito a Oscar Pichumán por compartir el código para este índice).

In [59]:
# ================================================================
# ÍNDICE DE COMORBILIDAD DE CHARLSON — IMPLEMENTACIÓN PROPIA
# Referencia: Quan et al. (2011) Med Care, 49(6), 612–617.
#             Charlson et al. (1987) J Chronic Dis, 40(5), 373–383.
# ================================================================

# Mapeamos prefijos ICD-10 (3 caracteres) a la categoría Charlson.
# Cada categoría tiene un peso según el modelo original de Charlson
# actualizado por Quan et al. (2011).

CHARLSON_ICD10 = {
    # ── Peso 1 ──────────────────────────────────────────────
    # Infarto agudo de miocardio
    'I21': ('MI', 1), 'I22': ('MI', 1), 'I25': ('MI', 1),
    # Insuficiencia cardíaca congestiva
    'I50': ('CHF', 1),
    # Enfermedad vascular periférica
    'I70': ('PVD', 1), 'I71': ('PVD', 1), 'I73': ('PVD', 1),
    'I77': ('PVD', 1), 'I79': ('PVD', 1),
    # Enfermedad cerebrovascular
    'G45': ('CVD', 1), 'G46': ('CVD', 1),
    'I60': ('CVD', 1), 'I61': ('CVD', 1), 'I62': ('CVD', 1),
    'I63': ('CVD', 1), 'I64': ('CVD', 1), 'I65': ('CVD', 1),
    'I66': ('CVD', 1), 'I67': ('CVD', 1), 'I68': ('CVD', 1),
    'I69': ('CVD', 1),
    # Demencia
    'F00': ('DEM', 1), 'F01': ('DEM', 1), 'F02': ('DEM', 1),
    'F03': ('DEM', 1), 'G30': ('DEM', 1), 'G31': ('DEM', 1),
    # Enfermedad pulmonar crónica (EPOC)
    'J40': ('COPD', 1), 'J41': ('COPD', 1), 'J42': ('COPD', 1),
    'J43': ('COPD', 1), 'J44': ('COPD', 1), 'J45': ('COPD', 1),
    'J46': ('COPD', 1), 'J47': ('COPD', 1), 'J60': ('COPD', 1),
    'J61': ('COPD', 1), 'J62': ('COPD', 1), 'J63': ('COPD', 1),
    'J64': ('COPD', 1), 'J65': ('COPD', 1), 'J66': ('COPD', 1),
    'J67': ('COPD', 1),
    # Enfermedad del tejido conectivo
    'M05': ('RHD', 1), 'M06': ('RHD', 1), 'M32': ('RHD', 1),
    'M33': ('RHD', 1), 'M34': ('RHD', 1), 'M35': ('RHD', 1),
    # Úlcera péptica
    'K25': ('PUD', 1), 'K26': ('PUD', 1), 'K27': ('PUD', 1),
    'K28': ('PUD', 1),
    # Enfermedad hepática leve
    'B18': ('MLD', 1), 'K70': ('MLD', 1), 'K71': ('MLD', 1),
    'K73': ('MLD', 1), 'K74': ('MLD', 1), 'K76': ('MLD', 1),
    # Diabetes sin complicación
    'E10': ('DM', 1), 'E11': ('DM', 1), 'E12': ('DM', 1),
    'E13': ('DM', 1), 'E14': ('DM', 1),

    # ── Peso 2 ──────────────────────────────────────────────
    # Hemiplejia
    'G04': ('HEMI', 2), 'G81': ('HEMI', 2), 'G82': ('HEMI', 2),
    'G83': ('HEMI', 2),
    # Enfermedad renal moderada/severa
    'N03': ('REN', 2), 'N05': ('REN', 2), 'N18': ('REN', 2),
    'N19': ('REN', 2), 'N25': ('REN', 2),
    # Diabetes con complicación crónica
    'E102': ('DMC', 2), 'E112': ('DMC', 2), 'E122': ('DMC', 2),
    'E132': ('DMC', 2), 'E142': ('DMC', 2),
    # Tumor sólido sin metástasis
    'C00': ('TUM', 2), 'C01': ('TUM', 2), 'C02': ('TUM', 2),
    'C03': ('TUM', 2), 'C04': ('TUM', 2), 'C05': ('TUM', 2),
    'C06': ('TUM', 2), 'C07': ('TUM', 2), 'C08': ('TUM', 2),
    'C09': ('TUM', 2), 'C10': ('TUM', 2), 'C11': ('TUM', 2),
    'C12': ('TUM', 2), 'C13': ('TUM', 2), 'C14': ('TUM', 2),
    'C15': ('TUM', 2), 'C16': ('TUM', 2), 'C17': ('TUM', 2),
    'C18': ('TUM', 2), 'C19': ('TUM', 2), 'C20': ('TUM', 2),
    'C21': ('TUM', 2), 'C22': ('TUM', 2), 'C23': ('TUM', 2),
    'C24': ('TUM', 2), 'C25': ('TUM', 2), 'C26': ('TUM', 2),
    'C30': ('TUM', 2), 'C31': ('TUM', 2), 'C32': ('TUM', 2),
    'C33': ('TUM', 2), 'C34': ('TUM', 2), 'C37': ('TUM', 2),
    'C38': ('TUM', 2), 'C39': ('TUM', 2), 'C40': ('TUM', 2),
    'C41': ('TUM', 2), 'C43': ('TUM', 2), 'C45': ('TUM', 2),
    'C46': ('TUM', 2), 'C47': ('TUM', 2), 'C48': ('TUM', 2),
    'C49': ('TUM', 2), 'C50': ('TUM', 2), 'C51': ('TUM', 2),
    'C52': ('TUM', 2), 'C53': ('TUM', 2), 'C54': ('TUM', 2),
    'C55': ('TUM', 2), 'C56': ('TUM', 2), 'C57': ('TUM', 2),
    'C58': ('TUM', 2), 'C60': ('TUM', 2), 'C61': ('TUM', 2),
    'C62': ('TUM', 2), 'C63': ('TUM', 2), 'C64': ('TUM', 2),
    'C65': ('TUM', 2), 'C66': ('TUM', 2), 'C67': ('TUM', 2),
    'C68': ('TUM', 2), 'C69': ('TUM', 2), 'C70': ('TUM', 2),
    'C71': ('TUM', 2), 'C72': ('TUM', 2), 'C73': ('TUM', 2),
    'C74': ('TUM', 2), 'C75': ('TUM', 2), 'C76': ('TUM', 2),
    'C97': ('TUM', 2),
    # Leucemia
    'C91': ('LEU', 2), 'C92': ('LEU', 2), 'C93': ('LEU', 2),
    'C94': ('LEU', 2), 'C95': ('LEU', 2),
    # Linfoma
    'C81': ('LYM', 2), 'C82': ('LYM', 2), 'C83': ('LYM', 2),
    'C84': ('LYM', 2), 'C85': ('LYM', 2), 'C88': ('LYM', 2),
    'C90': ('LYM', 2),

    # ── Peso 3 ──────────────────────────────────────────────
    # Enfermedad hepática moderada/severa
    'K70': ('MSLD', 3), 'K72': ('MSLD', 3), 'K76': ('MSLD', 3),

    # ── Peso 6 ──────────────────────────────────────────────
    # Tumor sólido con metástasis
    'C77': ('METS', 6), 'C78': ('METS', 6),
    'C79': ('METS', 6), 'C80': ('METS', 6),
    # VIH / SIDA
    'B20': ('AIDS', 6), 'B21': ('AIDS', 6),
    'B22': ('AIDS', 6), 'B24': ('AIDS', 6),
}

print(f"Mapa Charlson cargado: {len(CHARLSON_ICD10)} prefijos ICD-10 mapeados")
print(f"Categorías únicas: {len(set(v[0] for v in CHARLSON_ICD10.values()))}")
indice_charlson = {key:val[1] for key,val in CHARLSON_ICD10.items()}

Mapa Charlson cargado: 169 prefijos ICD-10 mapeados
Categorías únicas: 19


In [60]:
results = []
for col in diag_cols:
    result = grd[col].str[:3].map(indice_charlson).fillna(0)
    results.append(result)
grd['INDICE_CHARLSON'] = pd.concat(results, axis=1).sum(axis=1)

In [61]:
# con un enfoque alejado de los diagnosticos y procedimientos particulares, se deja la
# mínima cantidad de columnas para estos valores
grd.drop(columns=(diag_cols[2:] + proced_cols[2:]), inplace=True)

Para guardar espacio, la librería de pandas tiene un tipo de archivo de terminación .parquet, que pesa menos y se lee mucho más rápido que cargar todos los CSV/TXT cada vez (indirectamente reduce la memoria RAM utilizada, dado que se carga el archivo final, y no todos los intermedios y sus variables asociadas, que pueden seguir consumiendo memoria).

In [65]:
# esto resulta en un archivo, dentro del mismo directorio actual, con el nombre 'grd_19-24.parquet'
grd.to_parquet('grd_19-24.parquet')

En caso de querer continuar desde acá el notebook, se puede cerrar (lo que borra las variables cargadas hasta el momento) y continuar desde aquí, ya que se guarda el dataset

In [ ]:
# import pandas as pd
# grd = pd.read_parquet('grd_19-24.parquet')

Como nuestro enfoque es en el acceso a hospitalización compleja, se filtran datos, considerando severidad moderada y mayor, o 2 y 3, y un peso grd mayor a 2.

In [66]:
# se consideran casos de peso > 2 y severidad 2 o 3, quedando con 6.4% del dataset
grd = grd[grd['IR_29301_PESO'] > 2]
grd = grd[(grd['IR_29301_SEVERIDAD'] == '2') | (grd['IR_29301_SEVERIDAD'] == '3')]
grd = grd.reset_index(drop=True)

Nuevamente, se guarda el archivo como parquet esta vez filtrado, y es posible empezar el código desde aquí, sin tener que cargar todos los archivos de nuevo.

In [67]:
grd.to_parquet('grd_ac_19-24.parquet')

In [ ]:
# import pandas as pd
# grd = pd.read_parquet('grd_ac_19-24.parquet')

Se realiza el cruce de cada dataset secundario con el GRD ya procesado y filtrado. Se renombran ciertas columnas y añaden variables clave utilizando esta nueva información.

In [68]:
# se realizan todos los merge para los que se preparo la base de grd
# merge con cie10.
grd = pd.merge(grd, cie10, on='DIAGNOSTICO1', how='left')
# merge con base establecimientos
grd = pd.merge(grd, estab, on='COD_ESTABLECIMIENTO',how='left')
# merge con proyeccion poblacion del INE
grd = pd.merge(grd, poblacion, on=['COD_REGION','COD_COMUNA','YEAR'], how='left')
# merge con camas hospitalarias por establecimiento
grd = pd.merge(grd, camas, on=['COD_ESTABLECIMIENTO','YEAR'], how='left')
# merge con indice pobreza multidimensional
grd = pd.merge(grd, pm, on='COD_COMUNA', how='left')
# se cambian ciertos nombres para ser mas descriptivos
grd.rename(columns={'COMUNA':'COMUNA_ESTAB', 'POB_MULTIDIM':'POB_MULT_ESTAB'}, inplace=True)
# la pobreza multidimensional tiene unos pocos valores vacios, se rellenan con la media
grd['POB_MULT_ESTAB'] = grd['POB_MULT_ESTAB'].fillna(grd['POB_MULT_ESTAB'].mean())
# se crea la variable booleana: comuna con establecimiento de alta complejidad (1/0 para si/no)
grd['COMUNA_CON_ESTAB_AC'] = grd['COMUNA_REGISTRADA'].isin(grd['COMUNA_ESTAB']).astype(int)
# variable fallecido es otra booleana que marca con 1 a pacientes fallecidos
grd["FALLECIDO"] = (grd["TIPOALTA"] == "FALLECIDO").astype(int)
# proporcion de camas criticas para cada establecimiento, respecto a camas totales
grd['CRITICAS_PROP'] = grd['CAMAS_CRITICAS'] / grd['CAMAS_TOTALES']
# se busca calcular la cantidad de camas según 1000 habitantes, por comuna. al haber diferentes
# poblaciones para la misma comuna en diferentes años, es necesario aislar estos datos por separado
estab_unicos_comuna = grd.drop_duplicates(subset=['COMUNA_ESTAB','COD_ESTABLECIMIENTO','YEAR'])
# se agrupa por comuna y año, lo que deja a varios establecimientos por comuna en cada grupo
comunas_unique = estab_unicos_comuna.groupby(['COMUNA_ESTAB','YEAR'],observed=True)
# se suman las camas de cada establecimiento, para cada grupo de comuna-año
camas_comuna = comunas_unique['CAMAS_TOTALES'].sum().reset_index(name='CAMAS_COMUNA')
# lo mismo para las camas criticas
criticas_comuna = comunas_unique['CAMAS_CRITICAS'].sum().reset_index(name='CRITICAS_COMUNA')
# se unen todos estos datos al grd, para tener las variables deseadas
grd = grd.merge(camas_comuna, on=['COMUNA_ESTAB','YEAR'], how='left')
grd = grd.merge(criticas_comuna, on=['COMUNA_ESTAB','YEAR'], how='left')
grd['CAMAS_1000HAB'] = grd['CAMAS_COMUNA']/grd['POB_COMUNA'] * 1000
grd['CRITICAS_1000HAB'] = grd['CRITICAS_COMUNA']/grd['POB_COMUNA'] * 1000
# se añade la tasa de mortalidad por comuna, calculada rapidamente con el promedio, ya que
# se utiliza la variable de 0 y 1 de fallecido, anteriormente creada
tasa_mortalidad_comuna = grd.groupby('COMUNA_ESTAB', observed=True)['FALLECIDO'].mean()
grd['TASA_MORTALIDAD_COMUNA'] = grd['COMUNA_ESTAB'].map(tasa_mortalidad_comuna)
# se transforman los dtypes de las nuevas columnas de poblacion
grd['POB_COMUNA'] = grd['POB_COMUNA'].astype('int32')
grd['POB_REGION'] = grd['POB_REGION'].astype('int32')
# se dropean nulos tras merge
grd = grd = grd.dropna(subset=[col for col in grd.columns if col not in cols_ignore]).reset_index(drop=True)
# se establecen ciertas variables categoricas que guardan espacio y reducen consumo ram del dataframe
categ_cols = ['COD_ESTABLECIMIENTO', 'COD_COMUNA','COD_REGION','YEAR','COMUNA_REGISTRADA']
for col in categ_cols:
    grd[col] = grd[col].astype('category')
# se muestra resultado final
display(grd.shape, grd.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 367895 entries, 0 to 367894
Data columns (total 52 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   COD_ESTABLECIMIENTO     367895 non-null  category      
 1   CIP_ENCRIPTADO          367895 non-null  string        
 2   SEXO                    367895 non-null  category      
 3   FECHA_NACIMIENTO        367895 non-null  datetime64[ns]
 4   PROVINCIA               367895 non-null  category      
 5   COMUNA_REGISTRADA       367895 non-null  category      
 6   PREVISION               367895 non-null  category      
 7   SERVICIO_SALUD          367895 non-null  category      
 8   TIPO_PROCEDENCIA        367895 non-null  category      
 9   TIPO_INGRESO            367895 non-null  category      
 10  FECHA_INGRESO           367895 non-null  datetime64[ns]
 11  FECHAALTA               367895 non-null  datetime64[ns]
 12  TIPOALTA                367895

(367895, 52)

None

Se guarda el resultado final como parquet, lo que facilita analizar este archivo rápidamente en el futuro.

In [69]:
grd.to_parquet("grd_procesado.parquet")